# 03 · Your own chat traces

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daseinlabs/open-jev/blob/main/cookbooks/03-your-own-chat-traces/cookbook.ipynb)


Run the cookbook 02 pipeline on **your** agent's logs, without writing a spec or an adapter. The
SDK derives both from the tool schemas in your traces.

```
traces.jsonl ──spec_from_tools──▶ decision dataset + spec.json ──train──▶ model ──decide──▶ next tool / argument / defer
```


**Two methods**, chosen with `METHOD` in the setup cell; everything else is the same:

- `'tod'` (default) fine-tunes [TOD](https://huggingface.co/parsecai/tod) (Gemma 4 12B + LoRA). Needs an
  NVIDIA GPU with 40 GB+ (A100, H100, L40S; on Colab an A100 runtime) or a 64 GB Apple silicon Mac.
  Downloads `google/gemma-4-12B-it` (24 GB, not gated) once.
- `'head'` trains small heads on a frozen Gemma 3 4B. Needs ~9 GB (any recent NVIDIA GPU, or any Apple
  silicon Mac). Downloads [`parsecai/gemma-3-4b-it`](https://huggingface.co/parsecai/gemma-3-4b-it)
  (8 GB, gated: accept the Gemma terms there first).

In [ ]:
METHOD = 'tod'  # 'tod': fine-tune TOD (40 GB+ GPU) | 'head': small heads on Gemma 3 4B (~9 GB)

# Setup: install the SDK if it's missing, log in to Hugging Face once (for the gated Gemma 3 backbone).
import importlib.util
if importlib.util.find_spec('openjev') is None:
    %pip install -q openjev-sdk

from huggingface_hub import get_token, login
if METHOD == 'head' and get_token() is None:
    login()

import json, shutil
from pathlib import Path
from openjev import compute
from openjev.tod.picker import finetune, release
from openjev.tod.rl import load_policy
from openjev.tod.train import train_dataset

def train(dataset, out):
    # fine-tune TOD, or train heads; then calibrate and report test accuracy per kind
    return finetune(dataset, out) if METHOD == 'tod' else train_dataset(dataset, None, out)

def fresh(path):
    """Remove a previous run's output, so every cell can be re-run."""
    shutil.rmtree(path, ignore_errors=True)
    return Path(path)

print('method:', METHOD, '| device:', compute.device() if METHOD == 'tod' or compute.backend() == 'torch' else 'metal (MLX)')

## 0. Traces

One JSON object per line, in the OpenAI chat format most agent stacks already log:

```json
{"id": "conv-123",
 "tools": [{"type": "function", "function": {"name": "cancel_order", "parameters": {...}}}],
 "messages": [{"role": "system", ...}, {"role": "user", ...},
              {"role": "assistant", "tool_calls": [...]}, {"role": "tool", ...}, ...],
 "outcome": 1.0}
```

`outcome` is optional. If you have one (resolved, CSAT, no escalation), use it to keep only good
conversations. **Point `TRACES` at your file.** Without one, this cell writes ShopDesk traces
(cookbook 02) so you can follow along.

In [ ]:
TRACES = Path('traces.jsonl')
if not TRACES.exists():
    from openjev.tod.envs.shopdesk import record_trace
    with TRACES.open('w') as stream:
        stream.writelines(json.dumps(record_trace(f'conv-{i:05d}', seed=i * 2)) + '\n' for i in range(1500))

traces = [json.loads(line) for line in TRACES.open() if line.strip()]
print(len(traces), 'traces; tools:', [t['function']['name'] for t in traces[0]['tools']])

## 1. Derive the spec, build the dataset, read the coverage report

| kind | candidates | from |
|---|---|---|
| `action` | every tool, plus `respond` (answer in text) | the `tools` list |
| `arg.<param>` for enum parameters | the enum values | the parameter schema |
| `arg.<param>` for ID parameters | IDs seen earlier in the conversation | `id_params` |

`id_params={'order_id': ['id']}` says: `order_id` is an ID, and its values also appear under the
key `id` in tool results and in JSON in the system prompt. Each candidate carries the fields found
next to it, e.g. `W1234 (item: blue kettle, status: pending)`, so the model can match "the kettle"
to the right ID.

**Read the coverage report before training.** It answers "how much of my agent can this model take over?":

- `skipped_fewer_than_2_candidates`: the ID was never seen before it was used. Add the key it appears under.
- `free_text_left_to_parent`: a parameter the agent writes (a search query, an email body). It stays with your big model.
- `recall_miss_rate > 0`: the agent used a value that wasn't among the candidates; the model can never choose those.

In [ ]:
from openjev.tod import build_dataset
from openjev.tod.chat_traces import episodes_from_traces, spec_from_tools

RUN = Path('runs/mine')
spec = spec_from_tools('my-agent', traces[0]['tools'], id_params={'order_id': ['id']})
episodes, coverage = episodes_from_traces(traces, spec)
dataset = build_dataset(spec, episodes, min_outcome=1.0)

DATASET = fresh(RUN/'dataset')
dataset.write(DATASET)
spec.save(RUN/'spec.json')  # serving rebuilds the identical spec from this file
print(json.dumps(dict(kinds=list(spec.kinds), parameter_coverage=coverage,
                      recall_miss_rate=dataset.report['recall_miss_rate'], splits=dataset.report['splits']), indent=1))

## 2. Train

The same step as cookbooks 01 and 02. ID selection is the harder kind,
with lower confidence; calibrating per kind lets you give it a higher deferral threshold.

In [ ]:
MODEL = fresh(RUN/'model')
result = train(DATASET, MODEL)
for kind, r in result['test'].items():
    print(f"{kind:13} accuracy {r['accuracy']:.1%}   mean confidence {r['mean_confidence']:.2f}   n={r['n']}")

## 3. Decide the next step in a live conversation

Render the conversation exactly as training did (`spec.state(messages)`) and ask for the next
`action`. If the chosen tool takes an ID, ask for `arg.<param>` with `dict(state, next_call=tool)`.
When `defer` is true, let your big model take the turn.

Here the customer wants to cancel a yoga mat, and the lookup says it's pending:

In [ ]:
from openjev.tod.chat_traces import ChatTraceSpec

profile = {'name': 'Ivan', 'orders': [{'id': 'W1125', 'item': 'yoga mat'}, {'id': 'W8190', 'item': 'cast-iron pan'},
                                      {'id': 'W9089', 'item': 'backpack'}]}
messages = [
    {'role': 'system', 'content': 'You are ShopDesk, a retail support agent. Look orders up before acting on them.\n'
                                  'Customer profile: ' + json.dumps(profile)},
    {'role': 'user', 'content': 'Changed my mind about the yoga mat, can you cancel it?'},
    {'role': 'assistant', 'content': None, 'tool_calls': [{'id': 'call_0', 'type': 'function', 'function': {
        'name': 'get_order', 'arguments': json.dumps({'order_id': 'W1125'})}}]},
    {'role': 'tool', 'tool_call_id': 'call_0', 'content': json.dumps({'id': 'W1125', 'item': 'yoga mat', 'status': 'pending'})},
]

spec = ChatTraceSpec.load(RUN/'spec.json')
policy = load_policy(MODEL, spec)  # refuses a model trained with a different spec
state = spec.state(messages)
action = policy.decide(state, 'action', threshold=0.8)
print('next tool:', action.choice_id, f'(confidence {action.confidence:.2f})', 'DEFER' if action.defer else '')
if not action.defer and 'arg.order_id' in spec.kinds and action.choice_id != 'respond':
    order = policy.decide(dict(state, next_call=action.choice_id), 'arg.order_id', threshold=0.8)
    print('order:', order.choice_id, f'(confidence {order.confidence:.2f})', 'DEFER' if order.defer else '')

## Getting good results

- **Volume:** a few thousand decision rows per kind is a good start.
- **Label quality beats quantity:** filter by outcome when you can. The model learns whatever your agent did, mistakes included.
- **When traces aren't enough:** if your agent is systematically wrong somewhere, imitation can't
  fix it. Build an `Environment` and use RL ([cookbook 02](../02-shopdesk-traces-to-rl/cookbook.ipynb), step 5).